## Integration Testing

This notebook covers **Integration Testing** with [xUnit](https://xunit.net), [FluentAssertions](https://fluentassertions.com) and [Playwright](https://playwright.dev/dotnet).

In this notebook, we will integration test some microservices in the Flixtube project.

- First we will examine existing integration tests for the `Metadata` microservice.
  - Writing integration tests for the `History` microservice will be left as an exercise.
- Next we will examine existing integration tests for the `Gateway` microservice.
  - Writing additional integration test for the `Gateway` microservice will be left as an exercise.

---

### Install the Playwright dotnet Tool

Let's make sure the Playwright dotnet Tool is installed globally on your computer:

- Open VSCode's terminal (Windows/Linux: `Ctrl + J`, Mac: `Cmd + J`), or via the main menu `Terminal -> New Terminal`.
- Execute the command below in your terminal:

  ```bash
  dotnet tool install --global Microsoft.Playwright.CLI
  ```

  You can also run the cell below to run the command via your operating system's shell.

In [1]:
!dotnet tool install --global Microsoft.Playwright.CLI

Tool 'microsoft.playwright.cli' is already installed.


---

### Install the Playwright VSCode Extension

<style>
    .container {
        width: 98%;
        margin-left: 0; /* Push the container to the left */
        margin-right: auto; 
    }
    .text-image {
        margin-bottom: 35px; /* Space between sections */
        overflow: hidden; /* Ensure image stays within the container */
    }
    .text {
        text-align: justify; /* Justify the text for better readability */
    }
    .image {
        float: right; /* Float the image to the right */
        margin-left: 25px; /* Space between image and text */
        margin-bottom: 10px; /* Space between image and text */
        max-width: 50%; /* Limit image size */
        height: auto; /* Maintain aspect ratio */
    }
</style>

<div class="container">
    <div class="text-image">
        <img class="image" src="notebook_images/playwright-extension.png">
        <div class="text">
            <p>
                Let's make sure the <a href="https://marketplace.visualstudio.com/items?itemName=ms-playwright.playwright">Playwright VSCode Extension</a> is installed:
            </p>
            <ul>
                <li>Click the <img src="notebook_images/extensions-view-icon.png" /> icon for the Extensions View in VSCode's <b>Activity Bar</b>.</li>
                <li>Search for the <b>Playwright Tests for VSCode</b> extension in the Search Bar.</li>
                <li>Select the <b>Playwright Tests for VSCode</b> extension, and click the <b>Install</b> button (if it isn't already installed).</li>
            </ul>
        </div>
    </div>
</div>

<div class="container">
    <div class="text-image">
        <img class="image" src="notebook_images/playwright-tool.png">
        <div class="text">
            <ul>
                <li>Click the Test Explorer icon <img src="notebook_images/test-explorer-view-icon.png"/> in the <b>Activity Bar</b>.</li>
                <li>You should now see a <b>PLAYWRIGHT</b> section below the <b>TEST EXPLORER</b> in the <b>Primary Side Bar</b>.</li>
            </ul>
        </div>
    </div>
</div>

You can also run the cell below to install the extension.

In [2]:
!code --install-extension ms-playwright.playwright --force

Installing extensions...
Extension 'ms-playwright.playwright' is already installed.


---

### Integration Testing Basics

In its simplest form, integration testing is just unit testing without any mocks.

- In **Unit Testing**, we want to **exclude all dependencies (services)** when testing the **Subject Under Test (SUT)**.
  - We want to isolate the SUT from its dependencies (services), such as accessing a database, or calling an external REST API.
- In **Integration Testing**, we want to **include all dependencies (services)** when testing the **Subject Under Test (SUT)**.
  - We want to the SUT to use its dependencies (services), such as accessing a database, or calling an external REST API.

So, to turn the unit tests we examined for our `MetadataController` and `GatewayController` in the previous notebook into integration test, we would simply remove all mocks and use the actual dependencies instead.

Although, a more interesting integration test, when testing a REST API controller, is to actually call the REST API with HTTP Requests and receive HTTP Responses.

- This adds an additional layer to the integration tests of the REST API controller.
- We could manually create an [HTTPClient](https://learn.microsoft.com/en-us/dotnet/api/system.net.http.httpclient?view=net-9.0) and use this to send HTTP Requests to our running `Flixtube.Metadata` or `Flixtube.Gateway` Web API project (which includes the HTTP pipeline in the tests).
- But a better approach is to use [Playwright](https://playwright.dev/dotnet) to do this (which also includes the HTTP pipeline in the tests).

Let's see how we can use [Playwright](https://playwright.dev/dotnet), together with [xUnit](https://xunit.net) and [FluentAssertions](https://fluentassertions.com), to integration test our `Flixtube.Metadata` and `Flixtube.Gateway` microservices. We'll start with the `Flixtube.Metadata` microservice.

---

### A quick recap of the `Metadata` microservice

In the previous notebook, we examined the `Metadata` microservice (see `flixtube -> Flixtube.Metadata  -> Flixtube.Metadata`):

- It is an ASP.NET Web API project with the following NuGet packages installed:
  - `Microsoft.EntityFrameworkCore`
  - `Microsoft.EntityFrameworkCore.Design`
  - `Microsoft.EntityFrameworkCore.Tools`
  - `Microsoft.EntityFrameworkCore.SqlServer`
  - `Microsoft.Extensions.Configuration`
  - `RabbitMQ.Client`
- The `Entities` folder defines the entity class `Video` with attributes `Id` and `Name`.
- The `Data` folder defines EFCore-related classes (`ApplicationDbContext`, `ApplicationDbContextFactory`, `SeedData`).
- The `Repositories` folder defines repository and unit of work classes (`Repository`, `VideoRepository`, `UnitOfWork`) and interfaces (`IRepository`, `IVideoRepository`, `IUnitOfWork`).
- The `Messages` folder defines the class `VideoUploadedMessage` used to deserialize a RabbitMQ message.
- The `Services` folder defines the `RabbitMqSubscriberService` class used to receive events (messages) published to a RabbitMQ exchange called `uploaded`.
- The `Program` class reads in environment variables, adds services to the Web API's service container, uses the `SeedData` class to ensure the SQL Server database is created (and seeded), configures the HTTP Request/Response pipeline, and starts the microservice listening on a specific port.
- The `Controllers` folder contains the class `MetadataController` that dependency injects services (added to the Web API's service container in the `Program` class) via its constructor, reads the configured environment variable values, and defines a number of REST API endpoints. 

---

### A quick recap of the `MetadataController`

In VSCode's explorer:

- Expand the folder `flixtube -> Flixtube.Metadata -> Flixtube.Metadata -> Controller`.
- Right-click the file `MetadataController.cs` and choose `Open to the side`.

The `MetadataController` accepts HTTP requests on the `/` route:

- `ILogger<MetadataController>` (for logging), `IConfiguration` (for reading configured environment variables) and `IUnitOfWork` (for communicating with the database) are dependency injected via the constructor.
- Notice the `/health` route which can be used by another service to check if the `Metadata` service is responding (i.e. a health check).
- The `/videos` GET route retrives and returns a list of video metadata from the database.
- The `/videos/{id}` GET route retrives and returns metadata for one specific video from the database.
- The `/video` POST route adds metadata for a video to the database.
- The `/video/{id}` DELETE route removes metadata for a specific video from the database.

What did we need to mock to **unit test** the `MetadataController` in the previous notebook?

- We needed to mock the services `ILogger<MetadataController>`, `IConfiguration` and `IUnitOfWork` that are dependency injected via the `MetadataController`'s constructor.
- We did this by creating fake versions of any methods called on these services by our SUT's code (using the `Moq` library).
- This prevented the actual services from being called (instead the fake/mocked versions were used).

What do we need to mock to **integration test** the `MetadataController`?

- **NOTHING!**
- In **integration testing** we want to test our SUT **together with (integrated with)** its dependencies (services).
  - We want to access actual databases (although a test database, not the production database).
  - We want to make HTTP calls to other microservices.
  - We want to publish messages to, and subscribe to, RabbitMQ exchanges and channels. 
  - etc.
- In this respect:
  - There is a lot less work to do in authoring **integration tests** compared to **unit tests** (we don't have to mock anything).
  - We test a larger part of our system (the integration of multiple components, i.e. if they actually work together).

---

### Integration Testing `Flixtube.Metadata`

In VSCode's explorer:

- Expand the folder `flixtube -> Flixtube.Metadata -> Flixtube.Metadata.IntegrationTests`.
- Right-click the file `Flixtube.Metadata.IntegrationTests.csproj` and choose `Open to the side`.
- We see that this is an `xunit` project, to which we have added:
  - The `Microsoft.Playwright`, `FluentAssertions` and `Microsoft.Extensions.Configuration.Json` NuGet packages.
  - A project reference to the `Flixtube.Metadata` project.
    - Note that this project reference is only created here to access the Video entity class in the `Flixtube.Metadata` project (we should,'t really need to have any project references for integration tests, as opposed to unit tests).
  - A `<CopyToOutputDirectory>` precompiler directive which copies our `appsettings.json` file from our project folder to the output directory (i.e. `bin/Debug/net9.0` for a debug build, and `bin/Release/net9.0` for a release build) when the project is compiled (built).
- Right-click the file `appsettings.json` and choose `Open to the side`.
  - Notice the setting `TestSettings.ApiBaseUtl` with the value `http://localhost:80`.
  - This is the scheme (`http`), domain (`localhost`) and port (`80`) used to access the `Flixtube.Metadata` microservice once it is running.

In VSCode's explorer:

- Right-click the file `PlaywrightFixture.cs` and choose `Open to the side`.
- This file defines the class `PlaywrightFixture` that implements the interface `IAsyncLifetime`.
  - An instance of this class will be dependency injected into our test classes, and used as a *common fixture* for our tests.
- The class declares three public properties of type `IConfiguration`, `IPlaywright` and `IAPIRequestContext`.
- The method `InitializeAsync()` initializes the three public properties.
  - `IConfiguration Configuration` holds configuration values that we read in from `appsettings.json` (in this case the Base Url to our microservice).
  - `IPlaywright Playwright` holds a Playwright instance (this is the main class defined in the `Microsoft.Playwright` NuGet package).
  - `IAPIRequestContext ApiContext` holds an ApiRequestContext instance (this is a Playwright class in the `Microsoft.Playwright` NuGet package used to send HTTP Requests and receive HTTP Responses, i.e. a Playwright version of an `HTTPClient`).
- The method `DisposeAsync()` cleans up (frees) allocated resources by disposing (destroying) the `ApiContext` and `Playwright` instances that were created in the `InitializeAsync()` method.

In VSCode's explorer:

- Right-click the file `MetadataControllerTests.cs` and choose `Open to the side`.
- This file contains an `xUnit` test class with test methods.
  - In this respect, it isn't any different from an `xUnit` test class with test methods in a unit test.
  - In can use a test fixture (just as we could use a test fixture in a unit test).
  - It adorns test methods with the `[Fact]` and/or `[Theory]` attributes (just as with unit tests).
  - It uses the `Assert` class or classes from `FluentAssertions` to assert test outcomes (just as in unit test).
  - It structures each test into the three sections Assert, Act, and Assert (just as in unit test).
- This file defines the class `MetadataControllerTests` that implements the generic interface `IClassFixture<PlaywrightFixture>`.
  - Notice the type parameter `PlaywrightFixture` which is the test fixture class defined in `PlaywrightFixture.cs`.
- At the top of the file, we declare a number of private attributes, which we set in the `MetadataControllerTests` constructor.
- The constructor:
  - Dependency injects `PlaywrightFixture` (an instance of our test fixture class) and `ITestOutputHelper` that we can use to output debug messages to the console when running a test.
  - Extracts the `ApiContext` from the dependency injected `PlaywrightFixture` instance.
  - Defines `JsonSerializerOptions` to be used with the `JsonSerializer` class.

#### Integration Testing Metadata's HTTP endpoints

- In the method `HttpGet_Should_Return_All_Metadata()`:
  - We test the SUT's (i.e. the `Flixtube.Metadata` microservice's) `HTTP GET /videos` endpoint, which returns a list of videos.
  - In the `Arrange` section, we first set the `expected` result to two `Video` instances `expected1` and `expected2`, then we use the `HTTP POST /video` endpoint to send these two `Video` objects to the `Flixtube.Metadata` microservice.
    - Notice we are using the `PostAsync()` method on the `IAPIRequestContext` instance to call the `HTTP POST /video` endpoint, where we are using an instance of the `APIRequestContextOptions` class and its `DataObject` property to set the payload (HTTP body) in the HTTP Request to `expected1` (a `Video` instance).
  - In the `Act` section, we call the SUT's (`Flixtube.Metadata` microservice's) `HTTP GET /videos` endpoint and store the response in a local variable.
    - Notice we are using the `GetAsync()` method on the `IAPIRequestContext` instance to call the `HTTP GET /videos` endpoint.
  - In the `Assert` section:
    - We use FluentAssertions to assert the response's `Status` is `200`.
    - We extract the response as text (`response.TextAsync()`) and use the `JsonSerializer` to deserialize the JSON text to a `List<Video>`. 
    - Then we verify the `expected` and `actual` list of videos is the same, with the same property values for each video.
    - We use the `ITestOutputHelper` to print some debug text to the console.
- In the method `HttpGet_ForSpecificVideo_Should_Return_Video_Metadata()`:
  - We test the SUT's (i.e. the `Flixtube.Metadata` microservice's) `HTTP GET /video/{id}` endpoint, which accepts a video's `id` and returns that video.
  - This method is tested similarly to the previous test method, but instead of comparing a list of videos, a single video is compared.
- In the method `HttpPost_Should_Return_Newly_Added_Metadata()`:
  - We test the SUT's (i.e. the `Flixtube.Metadata` microservice's) `HTTP POST /video` endpoint, which accepts a `Video` as the payload (HTTP body) and returns that same video.
  - This method is tested similarly to the previous test method, but accepts a Video instead of a video's Id as input.
- In the method `HttpDelete_Should_Return_Newly_Deleted_Metadata()`:
  - We test the SUT's (i.e. the `Flixtube.Metadata` microservice's) `HTTP DELETE /video/{id}` endpoint, which accepts a video's `id` and returns that same video.
  - This method is tested similarly to the `HTTP GET /video/{id}` endpoint above.

#### Running the Integration Tests with `dotnet test`

If we just wanted to integration test the `Metadata` microservice, we could:

- Create a solution file (with e.g. `dotnet new sln -n Flixtube.Metadata.sln`).
- Add `Flixtube.Metadata` and `Flixtube.Metadata.IntegrationTests` to the solution `Flixtube.Metadata.sln`.
- Right-click the solution file `Flixtube.Metadata.sln` an choose `Open Solution`.
- Start the microservice and make sure all its dependencies are avaialble (in this case, an SQL Server database and a RabbitMQ broker).
- Switch to the `Testing` view in VSCode and run the tests.
- Stop the microservice and clean up its dependencies if needed (in this case, an SQL Server database and a RabbitMQ broker).

A simpler way is just to use the `dotnet` CLI:

- Start the microservice and make sure all its dependencies are avaialble (in this case, an SQL Server database and a RabbitMQ broker).
- Run the tests
  - Alternative 1: Move into the folder that contains the solution file `Flixtube.Metadata.sln` and run `dotnet test`.
  - Alternative 2: Move into the folder that contains the project file `Flixtube.Metadata.IntegrationTests.csproj` and run `dotnet test`.
- Stop the microservice and clean up its dependencies if needed (in this case, an SQL Server database and a RabbitMQ broker).

Let's try the first alternative in the cells below.

**Note!**

- I am using **Docker Compose** to start the Metadata microservice, SQLServer and RabbitMQ in Docker containers (first cell below).
- Then I am using **Docker** to run the tests in the Metadata microservice Docker container (second cell below).
- Finally, I am using **Docker Compose** to stop the Metadata microservice, SQLServer and RabbitMQ Docker containers (third cell below).
- **We haven't gone through how to use Docker or Docker Compose yet, but we'll learn how to do this during the next workshop.**
- The `Flixtube.Metadata.sln` I am invoking with `dotnet test` inside the Metadata microservice Docker container contains three projects (`Flixtube.Metadata`, `Flixtube.Metadata.UnitTests` and `Flixtube.Metadata.IntegrationTests`) which is why both the unit tests and integration tests are run (second cell below).

Now, let's execute the cells below and observe the test results.

- As you can see, the final row in the output from the second cell below is:
  - `Passed!` (all tests passed)
  - `Failed: 0` (0 tests failed)
  - `Passed: 4` (4 tests passed)
  - `Skipped: 0` (no tests were skipped)
  - `Total: 4` (a total of 4 tests were found)
  - `Duration: 1 s` (the total testing time was 1 second)
  - `Flixtube.Metadata.IntegrationTests.dll (net9.0)` (the test assembly run)

In [4]:
!docker compose -f ../flixtube/compose/docker-compose-metadata-dev.yml --project-directory ../flixtube up -d --build --force-recreate

#0 building with "desktop-linux" instance using docker driver

#1 [metadata internal] load build definition from Dockerfile-dev
#1 transferring dockerfile: 454B 0.0s done
#1 DONE 0.0s

#2 [metadata internal] load metadata for mcr.microsoft.com/dotnet/sdk:9.0
#2 DONE 0.9s

#3 [metadata internal] load .dockerignore
#3 transferring context: 393B done
#3 DONE 0.0s

#4 [metadata internal] load build context
#4 transferring context: 5.77kB done
#4 DONE 0.0s

#5 [metadata 1/7] FROM mcr.microsoft.com/dotnet/sdk:9.0@sha256:84fd557bebc64015e731aca1085b92c7619e49bdbe247e57392a43d92276f617
#5 resolve mcr.microsoft.com/dotnet/sdk:9.0@sha256:84fd557bebc64015e731aca1085b92c7619e49bdbe247e57392a43d92276f617 0.0s done
#5 sha256:3cb0d12d309310b704601a48fdfcb3fbcc0a882c1957b4e1bb82b22d6ace979b 5.53kB / 5.53kB done
#5 sha256:b7f90be4bd5027190782556b3df6c604bedd9be254e4cb491f31e76f47fd2735 0B / 18.72MB 0.2s
#5 sha256:56676eeaaf44ac1365e729f9945a96438bf035b6cc5356cb84a7ae1865c5cf3d 3.28kB / 3.28kB 0.1s done

 Service metadata  Building
 Service metadata  Built
 Network flixtube_default  Creating
 Network flixtube_default  Created
 Volume "flixtube_sqlserver_data"  Creating
 Volume "flixtube_sqlserver_data"  Created
 Volume "flixtube_rabbit_data"  Creating
 Volume "flixtube_rabbit_data"  Created
 Container sqlserver  Creating
 Container rabbit  Creating
 Container sqlserver  Created
 Container rabbit  Created
 Container metadata  Creating
 Container metadata  Created
 Container sqlserver  Starting
 Container rabbit  Starting
 Container sqlserver  Started
 Container rabbit  Started
 Container metadata  Starting
 Container metadata  Started


In [7]:
!docker exec metadata dotnet test

  Determining projects to restore...
  All projects are up-to-date for restore.
  Flixtube.Metadata -> /src/Flixtube.Metadata/bin/Debug/net9.0/Flixtube.Metadata.dll
  Flixtube.Metadata.UnitTests -> /src/Flixtube.Metadata.UnitTests/bin/Debug/net9.0/Flixtube.Metadata.UnitTests.dll
Test run for /src/Flixtube.Metadata.UnitTests/bin/Debug/net9.0/Flixtube.Metadata.UnitTests.dll (.NETCoreApp,Version=v9.0)
VSTest version 17.12.0 (x64)

Starting test execution, please wait...
A total of 1 test files matched the specified pattern.

Passed!  - Failed:     0, Passed:     4, Skipped:     0, Total:     4, Duration: 1 s - Flixtube.Metadata.UnitTests.dll (net9.0)
  Flixtube.Metadata.IntegrationTests -> /src/Flixtube.Metadata.IntegrationTests/bin/Debug/net9.0/Flixtube.Metadata.IntegrationTests.dll
Test run for /src/Flixtube.Metadata.IntegrationTests/bin/Debug/net9.0/Flixtube.Metadata.IntegrationTests.dll (.NETCoreApp,Version=v9.0)
VSTest version 17.12.0 (x64)

Starting test execution, please wait...
A 

In [8]:
!docker compose -f ../flixtube/compose/docker-compose-metadata-dev.yml --project-directory ../flixtube down --rmi local --volumes

 Container metadata  Stopping
 Container metadata  Stopped
 Container metadata  Removing
 Container metadata  Removed
 Container sqlserver  Stopping
 Container rabbit  Stopping
 Container sqlserver  Stopped
 Container sqlserver  Removing
 Container sqlserver  Removed
 Container rabbit  Stopped
 Container rabbit  Removing
 Container rabbit  Removed
 Volume flixtube_sqlserver_data  Removing
 Image metadata:latest  Removing
 Volume flixtube_rabbit_data  Removing
 Network flixtube_default  Removing
 Image metadata:latest  Removed
 Volume flixtube_sqlserver_data  Removed
 Volume flixtube_rabbit_data  Removed
 Network flixtube_default  Removed


---

### Integration Test the `HistoryController` in the `Flixtube.History` Microservice

- As an exercise, try:
  - Adding an `xunit` project `Flixtube.History.IntegrationTests` to test the `Flixtube.History` microservice.
  - Add NuGet packages `Microsoft.Playwright`, `FluentAssertions` and `Microsoft.Extensions.Configuration.Json`.
  - Add a `appsettings.json` file with a setting for the `Flixtube.History` microservice's Base URL (http://localhost:80).
  - Create a project reference to `Flixtube.History`.
  - Add a class `PlaywrightFixture` (exact copy of the class used in the `Flixtube.Metadata.IntegrationTests` project above).
  - Add a class `HistoryControllerTests`, and write integration test for the `HistoryController`'s HTTP endpoints.
  - Run the same commands as in the three notebook cells above, but replace `-f ../flixtube/compose/docker-compose-metadata-dev.yml` with `-f ../flixtube/compose/docker-compose-history-dev.yml` in the first and third cells.
  - Note that the `Flixtube.History` microservice has a similar structure as the `Flixtube.Metadata` microservice, where both communicate with an SQL Server database and a RabbitMQ broker.

---

### A quick recap of the `Gateway` microservice

In the previous notebook, we examined the `Gateway` microservice (see `flixtube -> Flixtube.Gateway  -> Flixtube.Gateway`):

- It is an ASP.NET Web API project with no additional NuGet packages installed.
- The `Models` folder defines two classes:
  - `Video` with properties `Id` and `Name`.
  -  `ViewHistory` with properties `Id`, `VideoId` and `ViewedAt`.
- The `Program` class reads in environment variables, adds services to the Web API's service container (in this case 5 HTTPClient instances), configures the HTTP Request/Response pipeline, and starts the microservice listening on a specific port.
- The `Controllers` folder contains the class `GatewayController` that dependency injects services (added to the Web API's service container in the `Program` class) via its constructor, reads the configured environment variable values, and defines a number of REST API endpoints.

---

### A quick recap of the `GatewayController`

In VSCode's explorer:

- Expand the folder `flixtube -> Flixtube.Gateway -> Flixtube.Gateway -> Controller`.
- Right-click the file `GatewayController.cs` and choose `Open to the side`.

The `GatewayController` accepts HTTP requests on the `/api` route:

- `ILogger<GatewayController>` (for logging), `IConfiguration` (for reading configured environment variables) and `IHttpClientFactory` (for communicating with the other backend microservices) are dependency injected via the constructor.
- Notice the `/health` route which can be used by another service to check if the `Gateway` service is responding (i.e. a health check).

- The `/api/metadata` GET route retrives and returns a list of video metadata from the Metadata microservice.
- The `/api/metadata/{id}` GET route retrives and returns metadata for one specific video from the Metadata microservice.
- The `/api/metadata` POST route adds metadata for a video via the Metadata microservice.
- The `/api/metadata/{id}` DELETE route removes metadata for a specific video via the Metadata microservice.
- The `/api/history` GET route retrives and returns a list of view history from the History microservice.
- The `/api/video/{id}` GET route streams a specific video from the videoStreaming microservice.
- The `/api/video` POST route uploads a video file via the VideoUpload microservice.
- The `/api/video/{id}` DELETE route removes a video file via the VideoStorage microservice.

What did we need to mock to **unit test** the `GatewayController` in the previous notebook?

- We needed to mock the services `ILogger<GatewayController>`, `IConfiguration` and `IHttpClientFactory` that are dependency injected via the `GatewayController`'s constructor.
- We did this by creating fake versions of any methods called on these services by our SUT's code (using the `Moq` library).
- This prevented the actual services from being called (instead the fake/mocked versions were used).

What do we need to mock to **integration test** the `GatewayController`?

- **NOTHING!** (as was the case when integration testing the Metadata microservice above)

---

### Integration Testing `Flixtube.Gateway`

In VSCode's explorer:

- Expand the folder `flixtube -> Flixtube.Gateway -> Flixtube.Gateway.IntegrationTests`.
- Right-click the file `Flixtube.Gateway.IntegrationTests.csproj` and choose `Open to the side`.
- We see that this is an `xunit` project, to which we have added:
  - The `Microsoft.Playwright`, `FluentAssertions` and `Microsoft.Extensions.Configuration.Json` NuGet packages.
  - A project reference to the `Flixtube.Gateway` project.
    - Note that this project reference is only created here to access the Video and ViewHistory model classes in the `Flixtube.Gateway` project (we should,'t really need to have any project references for integration tests, as opposed to unit tests).
  - A `<CopyToOutputDirectory>` precompiler directive which copies our `appsettings.json` file from our project folder to the output directory (i.e. `bin/Debug/net9.0` for a debug build, and `bin/Release/net9.0` for a release build) when the project is compiled (built).
- Right-click the file `appsettings.json` and choose `Open to the side`.
  - Notice the setting `TestSettings.ApiBaseUtl` with the value `http://localhost:80`.
  - This is the scheme (`http`), domain (`localhost`) and port (`80`) used to access the `Flixtube.Gateway` microservice once it is running.

In VSCode's explorer:

- Right-click the file `PlaywrightFixture.cs` and choose `Open to the side`.
- This is an exact copy of the `PlaywrightFixture.cs` file used in the integration tests for the `Flixtube.Metadata` microservice above.

In VSCode's explorer:

- Right-click the file `GatewayControllerTests.cs` and choose `Open to the side`.
- This file has a similar structure as the `MetadataControllerTests.cs` file used when integration testing the `Flixtube.Metadata` microservice above.
- This file defines the class `GatewayControllerTests` that implements the generic interface `IClassFixture<PlaywrightFixture>`.
  - Notice the type parameter `PlaywrightFixture` which is the test fixture class defined in `PlaywrightFixture.cs`.
- At the top of the file, we declare a number of private attributes, which we set in the `GatewayControllerTests` constructor.
- The constructor:
  - Dependency injects `PlaywrightFixture` (an instance of our test fixture class) and `ITestOutputHelper` that we can use to output debug messages to the console when running a test.
  - Extracts the `ApiContext` from the dependency injected `PlaywrightFixture` instance.
  - Defines `JsonSerializerOptions` to be used with the `JsonSerializer` class.

#### Integration Testing Gateway's HTTP endpoints

- In the method `HttpGet_Should_Return_All_Metadata()`:
  - We test the SUT's (i.e. the `Flixtube.Gateway` microservice's) `HTTP GET /api/metadata` endpoint, which returns a list of videos.
  - In the `Arrange` section, we first set the `expected` result to two `Video` instances `expected1` and `expected2`, then we use the `HTTP POST /api/metadata` endpoint to send these two `Video` objects to the `Flixtube.Gateway` microservice.
    - Notice we are using the `PostAsync()` method on the `IAPIRequestContext` instance to call the `HTTP POST /api/metadata` endpoint, where we are using an instance of the `APIRequestContextOptions` class and its `DataObject` property to set the payload (HTTP body) in the HTTP Request to `expected1` (a `Video` instance).
  - In the `Act` section, we call the SUT's (`Flixtube.Gateway` microservice's) `HTTP GET /ap/metadata` endpoint and store the response in a local variable.
    - Notice we are using the `GetAsync()` method on the `IAPIRequestContext` instance to call the `HTTP GET /api/metadata` endpoint.
  - In the `Assert` section:
    - We use FluentAssertions to assert the response's `Status` is `200`.
    - We extract the response as text (`response.TextAsync()`) and use the `JsonSerializer` to deserialize the JSON text to a `List<Video>`. 
    - Then we verify the `expected` and `actual` list of videos is the same, with the same property values for each video.
    - We use the `ITestOutputHelper` to print some debug text to the console.
- In the method `HttpGet_ForSpecificVideo_Should_Return_Video_Metadata()`:
  - We test the SUT's (i.e. the `Flixtube.Gateway` microservice's) `HTTP GET /api/metadata/{id}` endpoint, which accepts a video's `id` and returns that video.
  - This method is tested similarly to the previous test method, but instead of comparing a list of videos, a single video is compared.
- In the method `HttpPost_Should_Return_Newly_Added_Metadata()`:
  - We test the SUT's (i.e. the `Flixtube.Gateway` microservice's) `HTTP POST /api/metadata` endpoint, which accepts a `Video` as the payload (HTTP body) and returns that same video.
  - This method is tested similarly to the previous test method, but accepts a Video instead of a video's Id as input.
- In the method `HttpDelete_Should_Return_Newly_Deleted_Metadata()`:
  - We test the SUT's (i.e. the `Flixtube.Gateway` microservice's) `HTTP DELETE /api/metadata/{id}` endpoint, which accepts a video's `id` and returns that same video.
  - This method is tested similarly to the `HTTP GET /api/metadata/{id}` endpoint above.

#### Running the Integration Tests with `dotnet test`

If we just wanted to integration test the `Gateway` microservice, we could:

- Create a solution file (with e.g. `dotnet new sln -n Flixtube.Gateway.sln`).
- Add `Flixtube.Gateway` and `Flixtube.Gateway.IntegrationTests` to the solution `Flixtube.Gateway.sln`.
- Right-click the solution file `Flixtube.Gateway.sln` an choose `Open Solution`.
- Start the microservice and make sure all its dependencies are avaialble (in this case, all the other backend microservices (Metadata, History, VideoUpload, VideoStreaming, VideoStorge) two SQL Server instances (for the Metadata and History microservices) and a RabbitMQ broker).
- Switch to the `Testing` view in VSCode and run the tests.
- Stop the microservice and clean up its dependencies if needed (all the other backend microservices (Metadata, History, VideoUpload, VideoStreaming, VideoStorge) two SQL Server instances (for the Metadata and History microservices) and a RabbitMQ broker).

A simpler way is just to use the `dotnet` CLI:

- Start the microservice and make sure all its dependencies are avaialble (in this case, all the other backend microservices (Metadata, History, VideoUpload, VideoStreaming, VideoStorge) two SQL Server instances (for the Metadata and History microservices) and a RabbitMQ broker).
- Run the tests
  - Alternative 1: Move into the folder that contains the solution file `Flixtube.Gateway.sln` and run `dotnet test`.
  - Alternative 2: Move into the folder that contains the project file `Flixtube.Gateway.IntegrationTests.csproj` and run `dotnet test`.
- Stop the microservice and clean up its dependencies if needed (in this case, all the other backend microservices (Metadata, History, VideoUpload, VideoStreaming, VideoStorge) two SQL Server instances (for the Metadata and History microservices) and a RabbitMQ broker).

Let's try the first alternative in the cells below.

**Note!**

- I am using **Docker Compose** to start the Gateway microservice, all other backend microservices (Metadata, History, VideoUpload, VideoStreaming, VideoStorage), two SQLServer instances, and RabbitMQ in Docker containers (first cell below).
- Then I am using **Docker** to run the tests in the Gateway microservice Docker container (second cell below).
- Finally, I am using **Docker Compose** to stop the Gateway microservice, all other backend microservices (Metadata, History, VideoUpload, VideoStreaming, VideoStorage), two SQLServer instances, and RabbitMQ containers (third cell below).
- **We haven't gone through how to use Docker or Docker Compose yet, but we'll learn how to do this during the next workshop.**
- The `Flixtube.Gateway.sln` I am invoking with `dotnet test` inside the Metadata microservice Docker container contains three projects (`Flixtube.Gateway`, `Flixtube.Gateway.UnitTests` and `Flixtube.Gateway.IntegrationTests`) which is why both the unit tests and integration tests are run (second cell below).

Now, let's execute the cells below and observe the test results.

- As you can see, the final row in the output from the second cell below is:
  - `Passed!` (all tests passed)
  - `Failed: 0` (0 tests failed)
  - `Passed: 4` (4 tests failed)
  - `Skipped: 0` (no tests were skipped)
  - `Total: 4` (a total of 4 tests were found)
  - `Duration: 1 s` (the total testing time was 1 second)
  - `Flixtube.Gateway.IntegrationTests.dll (net9.0)` (the test assembly run)

In [9]:
!docker compose -f ../flixtube/compose/docker-compose-gateway-dev.yml --project-directory ../flixtube up -d --build --force-recreate

#0 building with "desktop-linux" instance using docker driver

#1 [minio-storage internal] load build definition from Dockerfile-dev
#1 transferring dockerfile: 300B 0.0s done
#1 DONE 0.0s

#2 [metadata internal] load build definition from Dockerfile-dev
#2 transferring dockerfile: 454B 0.0s done
#2 DONE 0.1s

#3 [history internal] load build definition from Dockerfile-dev
#3 transferring dockerfile: 285B done
#3 DONE 0.1s

#4 [history internal] load metadata for mcr.microsoft.com/dotnet/sdk:9.0
#4 DONE 0.5s

#5 [history internal] load .dockerignore
#5 transferring context: 393B done
#5 DONE 0.0s

#6 [metadata internal] load .dockerignore
#6 transferring context: 393B done
#6 DONE 0.0s

#7 [minio-storage internal] load .dockerignore
#7 transferring context: 393B done
#7 DONE 0.0s

#8 [metadata internal] load build context
#8 DONE 0.0s

#9 [history internal] load build context
#9 DONE 0.0s

#10 [minio-storage internal] load build context
#10 DONE 0.0s

#11 [metadata 1/5] FROM mcr.micros

 Service metadata  Building
 Service minio-storage  Building
 Service history  Building
 Service metadata  Built
 Service minio-storage  Built
 Service video-upload  Building
 Service video-streaming  Building
 Service video-upload  Built
 Service video-streaming  Built
 Service history  Built
 Service gateway  Building
 Service gateway  Built
 Network flixtube_default  Creating
 Network flixtube_default  Created
 Volume "flixtube_sqlserver_data"  Creating
 Volume "flixtube_sqlserver_data"  Created
 Volume "flixtube_minio_data"  Creating
 Volume "flixtube_minio_data"  Created
 Volume "flixtube_rabbit_data"  Creating
 Volume "flixtube_rabbit_data"  Created
 Container sqlserver  Creating
 Container rabbit  Creating
 Container minio  Creating
 Container sqlserver  Created
 Container rabbit  Created
 Container history  Creating
 Container metadata  Creating
 Container minio  Created
 Container video-storage  Creating
 Container minio-mc  Creating
 Container video-storage  Created
 Containe

In [11]:
!docker exec gateway dotnet test

  Determining projects to restore...
  All projects are up-to-date for restore.
  Flixtube.Gateway -> /src/Flixtube.Gateway/bin/Debug/net9.0/Flixtube.Gateway.dll
  Flixtube.Gateway.UnitTests -> /src/Flixtube.Gateway.UnitTests/bin/Debug/net9.0/Flixtube.Gateway.UnitTests.dll
Test run for /src/Flixtube.Gateway.UnitTests/bin/Debug/net9.0/Flixtube.Gateway.UnitTests.dll (.NETCoreApp,Version=v9.0)
VSTest version 17.12.0 (x64)

Starting test execution, please wait...
A total of 1 test files matched the specified pattern.
  Flixtube.Gateway.IntegrationTests -> /src/Flixtube.Gateway.IntegrationTests/bin/Debug/net9.0/Flixtube.Gateway.IntegrationTests.dll
Test run for /src/Flixtube.Gateway.IntegrationTests/bin/Debug/net9.0/Flixtube.Gateway.IntegrationTests.dll (.NETCoreApp,Version=v9.0)

Passed!  - Failed:     0, Passed:     4, Skipped:     0, Total:     4, Duration: 5 s - Flixtube.Gateway.UnitTests.dll (net9.0)
VSTest version 17.12.0 (x64)

Starting test execution, please wait...
A total of 1 tes

In [12]:
!docker compose -f ../flixtube/compose/docker-compose-gateway-dev.yml --project-directory ../flixtube down --rmi local --volumes

 Container minio-mc  Stopping
 Container gateway  Stopping
 Container minio-mc  Stopped
 Container minio-mc  Removing
 Container minio-mc  Removed
 Container gateway  Stopped
 Container gateway  Removing
 Container gateway  Removed
 Container metadata  Stopping
 Container history  Stopping
 Container video-streaming  Stopping
 Container video-upload  Stopping
 Container video-streaming  Stopped
 Container video-streaming  Removing
 Container metadata  Stopped
 Container metadata  Removing
 Container history  Stopped
 Container history  Removing
 Container video-upload  Stopped
 Container video-upload  Removing
 Container video-streaming  Removed
 Container metadata  Removed
 Container history  Removed
 Container sqlserver  Stopping
 Container video-upload  Removed
 Container video-storage  Stopping
 Container rabbit  Stopping
 Container rabbit  Stopped
 Container rabbit  Removing
 Container rabbit  Removed
 Container video-storage  Stopped
 Container video-storage  Removing
 Container 

---

### Integration Test additional `GatewayController` endpoints in the `Flixtube.Gateway` Microservice

- As an exercise, try:
- In the class `GatewayControllerTests`, write additional integration test for the `GatewayController`'s HTTP endpoints.
- Run the same commands as in the three notebook cells above.

---

### Conclusion

This completes the introduction to integration testing with xUnit, FluentAssertions and Playwright in VSCode, where we have integration tested the Metadata and Gateway micoservices' HTTP endpoints.

Next, we will look at End-To-End Testing with Playwright in VSCode:

- Open the file `endtoendtesting.ipynb`.
- When the notebook opens in VSCode, click the text `Select Kernel` (top-right), and choose `Python Environments... => conda (Python 3.11) .conda/bin/python`.
- Now you can follow the instructions in the notebook.